# SOXX Series ID 申报检索来源能力

冻结两个 SEC EDGAR 搜索 URL，只看 S000004354 的 N-Q/NPORT-P 索引是否可用；不打开申报正文、行情、收益。


In [1]:
import json,hashlib,re,subprocess,time,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_soxx_series_filing_search_20261023';plan=json.loads((card/'plan.json').read_text())
assert plan['frozen'] and not (card/'attempt_started.json').exists()
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
raw=root/'data/snapshots/stage1_soxx_series_filing_search_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'filing_body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[]
for i,spec in enumerate(plan['requests_in_order']):
 state['GET_attempts']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 name=spec['form'].replace('-','_');body=raw/(name+'.html');headers=raw/(name+'.headers')
 started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(plan['transport']['connect_timeout_seconds']),'--max-time',str(plan['transport']['total_timeout_seconds']),'--max-filesize',str(plan['transport']['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['transport']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=60)
  fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  result={'form':spec['form'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:
  result={'form':spec['form'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:200]}
 result.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
 if body.exists():
  data=body.read_text(errors='replace');result['series_id_occurrences']=len(re.findall('S000004354',data,re.I));result['SOXX_occurrences']=len(re.findall(r'\bSOXX\b',data,re.I));result['accession_link_count']=len(re.findall(r'/Archives/edgar/data/1100663/[^\"\s<>]+-index\.htm',data,re.I));result['accession_numbers_unique']=len(set(re.findall(r'\b\d{10}-\d{2}-\d{6}\b',data)));result['requested_form_occurrences']=len(re.findall(re.escape(spec['form']),data,re.I));result['title']=next(iter(re.findall(r'<title[^>]*>(.*?)</title>',data,re.I|re.S)),None)
 receipts.append(result)
 if i+1<len(plan['requests_in_order']):time.sleep(plan['transport']['seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'receipt_sha256':sha(card/'capture_receipts.json')}
(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-series-search-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  out.add(card/'source_manifest.json',arcname='source_manifest.json');out.add(card/'capture_receipts.json',arcname='capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='series-search-readback-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:
   run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
qualified=any(x.get('http_status')==200 and x.get('accession_numbers_unique',0)>0 and (x.get('SOXX_occurrences',0)>0 or x.get('series_id_occurrences',0)>1) for x in receipts)
result={'status':'series_specific_EDGAR_search_index_visible_source_only' if qualified else 'series_search_unqualified_or_unavailable','GET_attempts':state['GET_attempts'],'responses':[{'form':x['form'],'http_status':x.get('http_status'),'raw_bytes':x['raw_bytes'],'series_id_occurrences':x.get('series_id_occurrences',0),'SOXX_occurrences':x.get('SOXX_occurrences',0),'accession_numbers_unique':x.get('accession_numbers_unique',0),'title':x.get('title')} for x in receipts],'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'filing_body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'SOXX_filings_identified':False}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','GET_attempts','responses','nas_archive_sha256')})


{'status': 'series_specific_EDGAR_search_index_visible_source_only', 'GET_attempts': 2, 'responses': [{'form': 'N-Q', 'http_status': 200, 'raw_bytes': 24498, 'series_id_occurrences': 2, 'SOXX_occurrences': 0, 'accession_numbers_unique': 27, 'title': 'EDGAR Search Results'}, {'form': 'NPORT-P', 'http_status': 200, 'raw_bytes': 25429, 'series_id_occurrences': 2, 'SOXX_occurrences': 0, 'accession_numbers_unique': 29, 'title': 'EDGAR Search Results'}], 'nas_archive_sha256': '14789c4727e1a518511e56dbc6112a975798a2dc0efc2def0e234de54f8bdc90'}


## 中文结论

三个 SEC series 专属索引 GET（两份 N-Q/NPORT-P，加一份涵盖 N-CSR/N-CSRS 的合并页）都已原字节 NAS 归档并恢复。零网络修复后，2011–2018 年有 32 条未修订报告期、5 条系列级修订候选；接受时钟来自原信托目录逐 accession 绑定。N-Q 15 条原信托 accession 全对上，2011-01-31 N-CSRS accession 已定位。0 正文、0 收益、0 因子、0 拟合、0 账户；后续须查原始正文和成员价格。
